## Import

In [1]:
import json, os, time, pickle, warnings
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

/usr/local/lib/python3.13/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup

In [2]:
SEED = 42
N_TRIALS = 30   # same Optuna budget for every model
INNER_K = 3     # inner folds used for tuning
os.makedirs("weights_V6_0", exist_ok=True)

In [3]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df = df[df["Income"] != 666666].copy()  # drop $666,666 income outlier (same treatment as Year_Birth sentinels)
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
df = df.dropna(subset=["Income"]).copy()  # 2213 rows (target must not be imputed)
print("cleaned:", df.shape, "| Income max:", f"{df['Income'].max():,.0f}", "| median:", f"{df['Income'].median():,.0f}")
REG_FEATURES = ["NumCatalogPurchases","MntMeatProducts","MntWines","NumWebVisitsMonth","MntSweetProducts",
    "MntFruits","NumWebPurchases","MntGoldProds","NumStorePurchases","MntFishProducts","Kidhome","Age","NumDealsPurchases"]
X_full = df[REG_FEATURES].values; y_raw = df["Income"].values.astype(float)
print(f"features: {len(REG_FEATURES)} dims (all numeric)")

cleaned: (2212, 28) | Income max: 162,397 | median: 51,371
features: 13 dims (all numeric)


## Dataloader

In [4]:
def prep_split(tr_idx, te_idx):
    """Standardise using the training rows only (no missing values left in the regression set)."""
    sc = StandardScaler().fit(X_full[tr_idx])
    return sc.transform(X_full[tr_idx]), sc.transform(X_full[te_idx]), y_raw[tr_idx], y_raw[te_idx]

# same 80/20 split as every earlier version (seed 42)
idx_all = np.arange(len(y_raw))
tr_idx, te_idx = train_test_split(idx_all, test_size=0.20, random_state=SEED)
Xtr, Xte, ytr, yte = prep_split(tr_idx, te_idx)
print(f"train {Xtr.shape} test {Xte.shape}")

train (1769, 13) test (443, 13)


## Model

In [5]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures, FunctionTransformer
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor,
                              HistGradientBoostingRegressor)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

def _scaled_y(est): return TransformedTargetRegressor(regressor=est, transformer=StandardScaler())

class OneFeature(FunctionTransformer):
    def __init__(self, j=0): super().__init__(func=None); self.j = j
    def fit(self, X, y=None): return self
    def transform(self, X): return X[:, [self.j]]

SPECS = {
 "Linear regression (1 feature)": dict(tier="curriculum",
    space=lambda t: dict(j=t.suggest_int("j", 0, len(REG_FEATURES) - 1)),
    make=lambda p, s: make_pipeline(OneFeature(p["j"]), LinearRegression())),
 "Multiple regression": dict(tier="curriculum", fixed=True, space=lambda t: {},
    make=lambda p, s: LinearRegression()),
 "Polynomial regression": dict(tier="curriculum",
    space=lambda t: dict(deg=t.suggest_int("deg", 2, 3), a=t.suggest_float("a", 1e-2, 1e4, log=True)),
    make=lambda p, s: make_pipeline(PolynomialFeatures(p["deg"], include_bias=False), StandardScaler(), Ridge(alpha=p["a"]))),
 "Decision tree": dict(tier="curriculum family",
    space=lambda t: dict(d=t.suggest_int("d", 2, 12), leaf=t.suggest_int("leaf", 1, 30)),
    make=lambda p, s: DecisionTreeRegressor(max_depth=p["d"], min_samples_leaf=p["leaf"], random_state=s)),
 "Random forest": dict(tier="curriculum family",
    space=lambda t: dict(n=t.suggest_int("n", 100, 400, step=100), d=t.suggest_int("d", 3, 16),
                         leaf=t.suggest_int("leaf", 1, 10), mf=t.suggest_categorical("mf", [1.0, 0.5, 0.33])),
    make=lambda p, s: RandomForestRegressor(n_estimators=p["n"], max_depth=p["d"], min_samples_leaf=p["leaf"],
                         max_features=p["mf"], n_jobs=1, random_state=s)),
 "ExtraTrees": dict(tier="curriculum family",
    space=lambda t: dict(n=t.suggest_int("n", 100, 400, step=100), d=t.suggest_int("d", 3, 16),
                         leaf=t.suggest_int("leaf", 1, 10), mf=t.suggest_categorical("mf", [1.0, 0.5, 0.33])),
    make=lambda p, s: ExtraTreesRegressor(n_estimators=p["n"], max_depth=p["d"], min_samples_leaf=p["leaf"],
                         max_features=p["mf"], n_jobs=1, random_state=s)),
 "Gradient boosting": dict(tier="curriculum family",
    space=lambda t: dict(n=t.suggest_int("n", 50, 300, step=50), lr=t.suggest_float("lr", 0.01, 0.3, log=True),
                         d=t.suggest_int("d", 2, 5), sub=t.suggest_float("sub", 0.6, 1.0)),
    make=lambda p, s: GradientBoostingRegressor(n_estimators=p["n"], learning_rate=p["lr"], max_depth=p["d"],
                         subsample=p["sub"], random_state=s)),
 "HistGradientBoosting": dict(tier="curriculum family",
    space=lambda t: dict(it=t.suggest_int("it", 50, 400, step=50), lr=t.suggest_float("lr", 0.01, 0.3, log=True),
                         leaf=t.suggest_int("leaf", 8, 64), l2=t.suggest_float("l2", 1e-3, 10, log=True)),
    make=lambda p, s: HistGradientBoostingRegressor(max_iter=p["it"], learning_rate=p["lr"], max_leaf_nodes=p["leaf"],
                         l2_regularization=p["l2"], random_state=s)),
 "kNN": dict(tier="curriculum family",
    space=lambda t: dict(k=t.suggest_int("k", 3, 51, step=2), w=t.suggest_categorical("w", ["uniform", "distance"]),
                         p=t.suggest_int("p", 1, 2)),
    make=lambda p, s: KNeighborsRegressor(n_neighbors=p["k"], weights=p["w"], p=p["p"])),
 "SVR (RBF)": dict(tier="curriculum family",
    space=lambda t: dict(C=t.suggest_float("C", 1e-1, 1e2, log=True), g=t.suggest_float("g", 1e-4, 1.0, log=True),
                         e=t.suggest_float("e", 0.01, 0.5, log=True)),
    make=lambda p, s: _scaled_y(SVR(C=p["C"], gamma=p["g"], epsilon=p["e"]))),
 "MLP (sklearn)": dict(tier="curriculum family",
    space=lambda t: dict(h1=t.suggest_int("h1", 16, 128, step=16), two=t.suggest_categorical("two", [False, True]),
                         a=t.suggest_float("a", 1e-5, 1e-1, log=True), lr=t.suggest_float("lr", 1e-3, 1e-2, log=True)),
    make=lambda p, s: _scaled_y(MLPRegressor(hidden_layer_sizes=(p["h1"], p["h1"] // 2) if p["two"] else (p["h1"],),
                         alpha=p["a"], learning_rate_init=p["lr"], max_iter=400, early_stopping=True,
                         n_iter_no_change=15, random_state=s))),
}

## Tuning

In [6]:
def inner_oof(spec, params, X, y, seed):
    """Out-of-fold predictions on the training rows (inner K-fold)."""
    oof = np.zeros(len(y))
    for a, b in KFold(INNER_K, shuffle=True, random_state=seed).split(X):
        est = spec["make"](params, seed)
        est.fit(X[a], y[a])
        oof[b] = est.predict(X[b])
    return oof

def tune(spec, X, y, seed):
    """Optuna (TPE) minimising RMSE of the inner out-of-fold predictions."""
    if spec.get("fixed"):
        return {}
    def objective(trial):
        return float(np.sqrt(mean_squared_error(y, inner_oof(spec, spec["space"](trial), X, y, seed))))
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=N_TRIALS)
    return spec["space"](optuna.trial.FixedTrial(study.best_params))

def fit_eval(spec, params, Xa, ya, Xb, yb, seed):
    est = spec["make"](params, seed)
    est.fit(Xa, ya)
    pred = est.predict(Xb)
    mse = mean_squared_error(yb, pred)
    return {"r2": r2_score(yb, pred), "rmse": float(np.sqrt(mse)), "mae": mean_absolute_error(yb, pred), "mse": mse}, pred

In [7]:
best_params = {}
for name, spec in SPECS.items():
    t0 = time.time()
    best_params[name] = tune(spec, Xtr, ytr, SEED)
    print(f"{name:32s} tuned in {time.time() - t0:6.1f}s | {best_params[name]}")
json.dump(best_params, open("weights_V6_0/regression_sklearn_best_params.json", "w"), indent=2, default=str)

Linear regression (1 feature)    tuned in    0.1s | {'j': 1}
Multiple regression              tuned in    0.0s | {}


Polynomial regression            tuned in    9.1s | {'deg': 2, 'a': 61.63020306936902}


Decision tree                    tuned in    0.5s | {'d': 6, 'leaf': 2}


Random forest                    tuned in   53.5s | {'n': 300, 'd': 14, 'leaf': 1, 'mf': 0.5}


ExtraTrees                       tuned in   36.1s | {'n': 100, 'd': 15, 'leaf': 1, 'mf': 0.5}


Gradient boosting                tuned in   30.2s | {'n': 250, 'lr': 0.10660679097432305, 'd': 4, 'sub': 0.6957079686260053}


HistGradientBoosting             tuned in  649.3s | {'it': 300, 'lr': 0.09843002647092639, 'leaf': 19, 'l2': 2.2099881622532864}


kNN                              tuned in    0.9s | {'k': 7, 'w': 'distance', 'p': 1}


SVR (RBF)                        tuned in    8.8s | {'C': 41.284521213658664, 'g': 0.0329714393599075, 'e': 0.1667655768229163}


MLP (sklearn)                    tuned in  128.5s | {'h1': 96, 'two': True, 'a': 0.0015375920235481753, 'lr': 0.001530574436550018}


## Training

In [8]:
results, scores = [], {}
for name, spec in SPECS.items():
    row, s = fit_eval(spec, best_params[name], Xtr, ytr, Xte, yte, SEED)
    scores[name] = s
    results.append(dict(model=name, tier=spec["tier"], selection="inner validation",
                        **{k: round(float(v), 4) for k, v in row.items()}, params=best_params[name]))
    print(f"{name:32s} R2 = {row['r2']:.4f}")

Linear regression (1 feature)    R2 = 0.4479
Multiple regression              R2 = 0.7244
Polynomial regression            R2 = 0.7546
Decision tree                    R2 = 0.6909


Random forest                    R2 = 0.7645


ExtraTrees                       R2 = 0.7783


Gradient boosting                R2 = 0.8429


HistGradientBoosting             R2 = 0.8531
kNN                              R2 = 0.7503


SVR (RBF)                        R2 = 0.8162


MLP (sklearn)                    R2 = 0.8148


## Evaluation

In [9]:
table = pd.DataFrame(results).sort_values("r2", ascending=False)
print(table.drop(columns=["params", "selection"]).to_string(index=False))
json.dump({"n_test": int(len(yte)), "seed": SEED, "n_trials": N_TRIALS, "inner_k": INNER_K, "models": results},
          open("weights_V6_0/regression_sklearn_test_metrics.json", "w"), indent=2, default=str)
np.savez("weights_V6_0/regression_sklearn_test_scores.npz", y_true=yte, **{k: v for k, v in scores.items()})

                        model              tier     r2       rmse        mae          mse
         HistGradientBoosting curriculum family 0.8531  8352.4987  5227.6145 6.976424e+07
            Gradient boosting curriculum family 0.8429  8637.8206  5225.7922 7.461195e+07
                    SVR (RBF) curriculum family 0.8162  9343.2452  5937.3117 8.729623e+07
                MLP (sklearn) curriculum family 0.8148  9380.3723  6312.8212 8.799139e+07
                   ExtraTrees curriculum family 0.7783 10261.6298  5606.7789 1.053010e+08
                Random forest curriculum family 0.7645 10577.1511  5978.9642 1.118761e+08
        Polynomial regression        curriculum 0.7546 10796.4093  6637.2179 1.165625e+08
                          kNN curriculum family 0.7503 10891.3077  6173.2013 1.186206e+08
          Multiple regression        curriculum 0.7244 11441.1156  7224.9036 1.308991e+08
                Decision tree curriculum family 0.6909 12117.0484  7706.2523 1.468229e+08
Linear reg